In [ ]:
# Day 1: inspect sources before training; never infer video order from SMOKE5K filenames.
from pathlib import Path
import json, collections, torch, zipfile
root=Path('/kaggle/input')
report={'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'sources':[str(p) for p in root.iterdir()],'checkpoints':[],'manifests':[],'archives':[]}
for p in root.rglob('best.pth'):
 c=torch.load(p,map_location='cpu',weights_only=True)
 report['checkpoints'].append({'path':str(p),'update':c.get('update'),'validation':c.get('validation'),'n_tensors':len(c.get('state_dict',{}))})
 del c
for p in root.rglob('manifest.json'):
 m=json.loads(p.read_text())
 report['manifests'].append({'path':str(p),'splits':{k:len(v) for k,v in m.items() if isinstance(v,list)},'examples':{k:v[:2] for k,v in m.items() if isinstance(v,list)}})
for p in root.rglob('*.zip'):
 with zipfile.ZipFile(p) as z:
  ns=z.namelist(); report['archives'].append({'path':str(p),'n_files':len(ns),'examples':ns[:12]})
report['temporal_status']='UNVERIFIED: sequence, camera and timestamp metadata required before temporal training.'
Path('/kaggle/working/day1_audit.json').write_text(json.dumps(report,indent=2,default=str))
print('DAY1_AUDIT',json.dumps(report,default=str),flush=True)


In [ ]:
import torch, subprocess
print('GPU_CHECK',torch.__version__,torch.cuda.is_available())
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)


In [ ]:
# Obtain the author's SmokeSeg archive and inspect labels before any temporal training.
from pathlib import Path
import subprocess,sys,json,zipfile,time,requests,re
from bs4 import BeautifulSoup
work=Path('/kaggle/working'); start=time.time()
print('DATA_PREP_START',flush=True)
subprocess.run([sys.executable,'-m','pip','-q','install','gdown'],check=True)
import gdown
archive=work/'SmokeSeg.zip'
if not archive.is_file():
 gdown.download(id='1AIHZ5PckjHGqeaG3VY5WZgH-Y-5lAzA0',output=str(archive),quiet=True)
assert archive.is_file()
with zipfile.ZipFile(archive) as z:
 names=z.namelist()
 print('SMOKESEG_ARCHIVE',len(names),names[:35],flush=True)
 textfiles=[n for n in names if n.lower().endswith(('.txt','.csv','.json'))]
 info={'archive_bytes':archive.stat().st_size,'n_files':len(names),'text_files':textfiles,'sample_names':names[:100],'metadata_samples':{n:z.read(n).decode('utf8',errors='replace')[:1200] for n in textfiles[:8]}}
 (work/'smokeseg_archive_audit.json').write_text(json.dumps(info,indent=2))
 print('SMOKESEG_METADATA',json.dumps(info),flush=True)
 for n in names:
  assert (work/'SmokeSeg-data'/n).resolve().is_relative_to((work/'SmokeSeg-data').resolve())
 z.extractall(work/'SmokeSeg-data')
print('DATA_PREP_DONE_SECONDS',time.time()-start,flush=True)


In [ ]:
# Audit real sequence identity; exclude any event whose frames were seen during SMOKE5K pretraining.
from pathlib import Path
from PIL import Image
import numpy as np,re,json,collections,hashlib
r=Path('/kaggle/working/SmokeSeg-data/SmokeSeg')
ims={p.stem:p for p in (r/'JPEGImages').iterdir() if p.suffix.lower() in ('.jpg','.png','.jpeg') and not p.name.startswith('.')}
masks={p.stem:p for p in (r/'SegmentationClass').iterdir() if p.suffix=='.png' and not p.name.startswith('.')}
cp=next(Path('/kaggle/input').rglob('fosp_smoke5k_14000/manifest.json')); old=json.loads(cp.read_text())
seen={x['id'] for k in ('train','val','test') for x in old[k]}
rows=[]; excluded=set(); values=collections.Counter()
for key,p in ims.items():
 if key not in masks or '__' not in key: continue
 seq,stem=key.rsplit('__',1); m=re.fullmatch(r'(\d{10})_([+-]\d+)',stem)
 if not m: continue
 event=seq.split('_')[0]
 if stem in seen: excluded.add(event)
 rows.append(dict(id=key,sequence=seq,event=event,timestamp=int(m[1]),offset=int(m[2]),image=str(p),mask=str(masks[key])))
for x in rows[:20]:
 with Image.open(x['mask']) as im: values.update(map(int,np.unique(np.asarray(im))))
seqs=collections.defaultdict(list)
for x in rows:
 if x['event'] not in excluded: seqs[x['sequence']].append(x)
pairs={'train':[],'val':[],'test':[]}
for seq,items in sorted(seqs.items()):
 items.sort(key=lambda x:x['timestamp']); event=items[0]['event']; h=int(hashlib.sha256(('42:'+event).encode()).hexdigest()[:8],16)%10
 split='test' if h==0 else 'val' if h==1 else 'train'
 for a,b in zip(items,items[1:]):
  if 0<b['timestamp']-a['timestamp']<=300:
   pairs[split].append(dict(**b,previous=a['image'],delta_seconds=b['timestamp']-a['timestamp']))
summary={'images':len(ims),'masks':len(masks),'parsed':len(rows),'excluded_pretraining_dates':sorted(excluded),'mask_values_sample':dict(values),'pairs':{k:len(v) for k,v in pairs.items()},'events':{k:len({x['event'] for x in v}) for k,v in pairs.items()},'limitation':'date grouping conservative; source IDs exclusion covers known IDs, not all visual near-duplicates; no negative frames yet'}
Path('/kaggle/working/temporal_manifest.json').write_text(json.dumps(pairs,indent=2))
Path('/kaggle/working/temporal_audit.json').write_text(json.dumps(summary,indent=2))
print('TEMPORAL_AUDIT',json.dumps(summary),flush=True)
print('EXAMPLES',json.dumps({k:v[:1] for k,v in pairs.items()}),flush=True)


In [ ]:
from pathlib import Path
import sys,os,subprocess,json
root=Path('/kaggle/input')
repo=next(root.rglob('FoSp-official/config/fosp_config.py')).parent.parent
mmcv=Path('/kaggle/temp/mmcv')
if not (mmcv/'mmcv').is_dir(): subprocess.run(['git','clone','--depth','1','--branch','v1.6.2','https://github.com/open-mmlab/mmcv.git',str(mmcv)],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','addict','yapf==0.40.1','easydict','timm==0.6.13','regex','setuptools<81'],check=True)
deps=next(root.rglob('fosp-deps/timm/__init__.py')).parent.parent
sys.path[:0]=[str(repo),str(repo/'inpainter_setting'),str(mmcv),str(deps)]
os.environ['PYTHONPATH']=os.pathsep.join(sys.path)
import torch,numpy as np
from mmcv import Config
from mmseg.models import build_segmentor
from inpainter_setting.torch_utils.ops import bias_act,upfirdn2d
from torch_utils.ops import bias_act as ba,upfirdn2d as uf
for op in (bias_act,upfirdn2d,ba,uf): op._init=lambda:False
cfg=Config.fromfile(str(repo/'config/fosp_config.py')); cfg.model.pretrained=None
model=build_segmentor(cfg.model).cuda().eval()
weights=next(root.rglob('fosp_smoke5k_14000/best.pth'))
c=torch.load(weights,map_location='cpu',weights_only=True); model.load_state_dict(c['state_dict'],strict=True); del c
print('FOSP_LOADED',torch.cuda.get_device_name(0),str(weights),flush=True)


In [ ]:
# Day-1 pilot: frozen FoSp + equal-capacity static/temporal residual heads.
# This is NOT the full learned background-memory method and NOT the official SmokeSeg protocol.
import time,random,json,shutil,traceback
from pathlib import Path
import numpy as np,torch
import torch.nn.functional as F
from PIL import Image
out=Path('/kaggle/working/day1_pilot');out.mkdir(exist_ok=True)
pairs=json.loads(Path('/kaggle/working/temporal_manifest.json').read_text())
assert all(pairs[k] for k in ('train','val','test'))
assert not ({x['event'] for x in pairs['train']} & {x['event'] for x in pairs['val']+pairs['test']})
SEED=42;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.set_num_threads(2)
model.eval().requires_grad_(False)
class ResidualHead(torch.nn.Module):
 def __init__(self):
  super().__init__();self.net=torch.nn.Sequential(torch.nn.Conv2d(9,24,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(24,24,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(24,1,1))
  torch.nn.init.zeros_(self.net[-1].weight);torch.nn.init.zeros_(self.net[-1].bias)
 def forward(self,x):return x[:,:1]+self.net(x)
heads={k:ResidualHead().cuda() for k in ('static','temporal')}
heads['temporal'].load_state_dict(heads['static'].state_dict())
opts={k:torch.optim.AdamW(h.parameters(),lr=1e-4,weight_decay=1e-4) for k,h in heads.items()}
mean=np.array([123.675,116.28,103.53],np.float32);std=np.array([58.395,57.12,57.375],np.float32)
cache={};cache_limit=3200
@torch.no_grad()
def image_features(path):
 if path in cache:return tuple(t.cuda().float() for t in cache[path])
 with Image.open(path) as im: rgb=np.asarray(im.convert('RGB').resize((512,512),Image.Resampling.BILINEAR),dtype=np.float32)
 x=torch.from_numpy(((rgb-mean)/std).transpose(2,0,1).copy())[None].cuda()
 state=np.random.get_state();np.random.seed(123)
 try: logit=model.encode_decode(x,[{}])
 finally:np.random.set_state(state)
 logit=F.interpolate(logit,size=(128,128),mode='bilinear',align_corners=False).clamp(-15,15)
 rgbsmall=F.interpolate(torch.from_numpy(rgb.transpose(2,0,1).copy())[None].cuda()/255,size=(128,128),mode='bilinear',align_corners=False)
 if len(cache)<cache_limit:cache[path]=(logit.cpu().half(),rgbsmall.cpu().half())
 return logit,rgbsmall
@torch.no_grad()
def features(row,flip=False):
 cur,rgb=image_features(row['image']);old,prev=image_features(row['previous'])
 diff=(rgb-prev).abs(); rel=torch.exp(-4*diff.mean(1,keepdim=True))
 temporal=torch.cat((cur,old,rgb,diff,rel),1)
 static=torch.cat((cur,torch.zeros_like(old),rgb,torch.zeros_like(diff),torch.zeros_like(rel)),1)
 with Image.open(row['mask']) as im:
  y=torch.from_numpy((np.asarray(im.convert('L').resize((128,128),Image.Resampling.NEAREST))>0).astype(np.float32))[None,None].cuda()
 if flip:static=static.flip(-1);temporal=temporal.flip(-1);y=y.flip(-1)
 return {'static':static,'temporal':temporal},y
@torch.no_grad()
def evaluate():
 sums={k:[] for k in ('fosp','static','temporal')}
 for i,row in enumerate(pairs['val']):
  xs,y=features(row); truth=y.bool(); logits={'fosp':xs['static'][:,:1],**{k:h(xs[k]) for k,h in heads.items()}}
  for k,z in logits.items():
   p=z.sigmoid()>0.4;tp=(p&truth).sum().item();fp=(p&~truth).sum().item();fn=(~p&truth).sum().item()
   sums[k].append([2*tp/max(2*tp+fp+fn,1),tp/max(tp+fp+fn,1)])
  if (i+1)%100==0:print('VAL_PROGRESS',i+1,len(pairs['val']),flush=True)
 return {k:{'f1':float(np.mean(v,axis=0)[0]),'smoke_iou':float(np.mean(v,axis=0)[1])} for k,v in sums.items()}
def save(payload,path):
 temp=path.with_suffix('.tmp');torch.save(payload,temp);temp.replace(path)
config={'method':'frozen FoSp + residual correction pilot','input_size':512,'head_and_eval_size':128,'threshold':0.4,'seed':42,'budget_seconds':10800,'max_updates':6000,'eval_every':500,'patience':5,'min_delta':0.001,'test_used':False,'limitations':['positive pairs only; cannot measure fog false alarms','coarse 128px pilot metrics not comparable to native-resolution SMOKE5K scores','camera overlap possible across dates; not unseen-camera evaluation','reliability is a fixed heuristic, not a learned memory controller']}
(out/'config.json').write_text(json.dumps(config,indent=2))
shutil.copy2('/kaggle/working/temporal_manifest.json',out/'manifest.json');shutil.copy2('/kaggle/working/temporal_audit.json',out/'data_audit.json')
started=time.monotonic();update=0;best={k:-1. for k in heads};history=[];stale=0
if (out/'last.pth').is_file():
 c=torch.load(out/'last.pth',map_location='cpu',weights_only=False)
 for k in heads:heads[k].load_state_dict(c['heads'][k]);opts[k].load_state_dict(c['optimizers'][k])
 update=c['update'];best=c['best'];history=c['history'];random.setstate(c['python_rng']);np.random.set_state(c['numpy_rng']);torch.set_rng_state(c['torch_rng']);torch.cuda.set_rng_state_all(c['cuda_rng']);stale=c.get('stale',0)
 print('RESUMED',update,flush=True)
else:
 initial=evaluate();history.append({'update':0,'validation':initial});print('PILOT_INITIAL',json.dumps(initial),flush=True)
 for k in heads:
  best[k]=initial[k]['f1'];save({'state_dict':heads[k].state_dict(),'update':0,'validation':initial[k],'config':config},out/('best_'+k+'.pth'))
def snapshot():
 save({'update':update,'heads':{k:h.state_dict() for k,h in heads.items()},'optimizers':{k:o.state_dict() for k,o in opts.items()},'best':best,'history':history,'stale':stale,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),'config':config},out/'last.pth')
 (out/'history.json').write_text(json.dumps(history,indent=2))
 (out/'status.json').write_text(json.dumps({'update':update,'elapsed_seconds':time.monotonic()-started,'best':best},indent=2))
try:
 while update<6000 and time.monotonic()-started<10200:
  row=random.choice(pairs['train']);xs,y=features(row,random.random()<0.5);losses={}
  for k,h in heads.items():
   opts[k].zero_grad(set_to_none=True);z=h(xs[k]);p=z.sigmoid();dice=1-(2*(p*y).sum()+1)/(p.sum()+y.sum()+1);loss=F.binary_cross_entropy_with_logits(z,y)+dice;loss.backward();opts[k].step();losses[k]=float(loss.detach())
  update+=1
  if update%25==0: print('PILOT_TRAIN',json.dumps({'update':update,'loss':losses,'elapsed':round(time.monotonic()-started,1),'cache':len(cache)}),flush=True)
  if update%100==0:snapshot()
  if update%500==0:
   scores=evaluate();history.append({'update':update,'validation':scores});improved=False
   for k in heads:
    if scores[k]['f1']>best[k]:
     improved |= scores[k]['f1']>best[k]+0.001;best[k]=scores[k]['f1'];save({'state_dict':heads[k].state_dict(),'update':update,'validation':scores[k],'config':config},out/('best_'+k+'.pth'))
   stale=0 if improved else stale+1;snapshot();print('PILOT_VALIDATION',json.dumps(history[-1]),flush=True)
   if stale>=5:print('EARLY_STOP',stale,flush=True);break
finally:
 snapshot();shutil.copy2(weights,out/'fosp_base_14000.pth');shutil.make_archive('/kaggle/working/forestwatch_day1_pilot','zip',out)
 print('PILOT_SAVED',update,'/kaggle/working/forestwatch_day1_pilot.zip',flush=True)


In [ ]:
# Inspect label encoding, alignment and native-resolution baseline BEFORE more training.
from PIL import Image,ImageDraw
import numpy as np,torch,json
from IPython.display import display
print('DIAG_START',flush=True)
print('HISTORY',json.dumps(history),flush=True)
canvas=Image.new('RGB',(960,6*264),'white');draw=ImageDraw.Draw(canvas)
rows=pairs['val'][::max(1,len(pairs['val'])//6)][:6];checks=[]
for i,row in enumerate(rows):
 with Image.open(row['image']) as im:rgb=im.convert('RGB'); original_size=im.size
 with Image.open(row['mask']) as im:raw=np.asarray(im);gray=np.asarray(im.convert('L')); info={'id':row['id'],'image_size':original_size,'mask_size':im.size,'mode':im.mode,'raw_values':np.unique(raw).tolist(),'gray_values':np.unique(gray).tolist(),'mask_fraction':float((gray>0).mean())}
 pixels=np.asarray(rgb.resize((512,512),Image.Resampling.BILINEAR),np.float32)
 x=torch.from_numpy(((pixels-mean)/std).transpose(2,0,1).copy())[None].cuda()
 state=np.random.get_state();np.random.seed(123)
 with torch.no_grad():z=model.encode_decode(x,[{}]);prob=F.interpolate(z,size=gray.shape,mode='bilinear',align_corners=False)[0,0].sigmoid().cpu().numpy()
 np.random.set_state(state);p=prob>0.4;y=gray>0;tp=(p&y).sum();fp=(p&~y).sum();fn=(~p&y).sum();info['native_f1']=float(2*tp/max(2*tp+fp+fn,1));info['pred_fraction']=float(p.mean());checks.append(info)
 arr=np.array(rgb);gt=arr.copy();pred=arr.copy();gt[y]=(gt[y]*.5+np.array([0,255,0])*.5).astype('uint8');pred[p]=(pred[p]*.5+np.array([255,150,0])*.5).astype('uint8')
 for j,img in enumerate((rgb,Image.fromarray(gt),Image.fromarray(pred))):canvas.paste(img.resize((320,240)),(j*320,i*264+24))
 draw.text((4,i*264+4),str(i)+' '+row['id']+' F1='+str(round(info['native_f1'],3)),fill='black')
print('LABEL_CHECKS',json.dumps(checks),flush=True);canvas.save(out/'diagnostic_preview.jpg');display(canvas)
(out/'diagnostics.json').write_text(json.dumps(checks,indent=2))


In [ ]:
# Compare the SAME predictions at 128 and native 512 resolution; validation only.
import json,time
for k in heads:
 c=torch.load(out/('best_'+k+'.pth'),map_location='cpu',weights_only=True);heads[k].load_state_dict(c['state_dict']);heads[k].eval()
res={k:[] for k in ('fosp_native','fosp_128','static_native','temporal_native')};empty128=0;zero_pred=0
for i,row in enumerate(pairs['val']):
 with Image.open(row['image']) as im:rgb=np.asarray(im.convert('RGB').resize((512,512),Image.Resampling.BILINEAR),np.float32)
 with Image.open(row['mask']) as im:target=np.asarray(im)>0
 x=torch.from_numpy(((rgb-mean)/std).transpose(2,0,1).copy())[None].cuda();state=np.random.get_state();np.random.seed(123)
 with torch.no_grad():logit=model.encode_decode(x,[{}]); native=F.interpolate(logit,size=target.shape,mode='bilinear',align_corners=False)[0,0].sigmoid().cpu().numpy();xs,y=features(row);preds={'fosp_native':native>0.4,'fosp_128':xs['static'][0,0].sigmoid().cpu().numpy()>0.4,**{k+'_native':F.interpolate(heads[k](xs[k]),size=target.shape,mode='bilinear',align_corners=False)[0,0].sigmoid().cpu().numpy()>0.4 for k in heads}}
 np.random.set_state(state);small=y[0,0].cpu().numpy()>0;empty128+=int(not small.any());zero_pred+=int(not preds['fosp_native'].any())
 for k,p in preds.items():
  t=small if k.endswith('128') else target;tp=int((p&t).sum());fp=int((p&~t).sum());fn=int((~p&t).sum());res[k].append({'f1':2*tp/max(2*tp+fp+fn,1),'iou':tp/max(tp+fp+fn,1),'tp':tp,'fp':fp,'fn':fn,'gt_pixels':int(t.sum())})
 if (i+1)%100==0:print('RESOLUTION_PROGRESS',i+1,flush=True)
summary={k:{'image_f1':float(np.mean([x['f1'] for x in v])),'image_iou':float(np.mean([x['iou'] for x in v]))} for k,v in res.items()}
summary.update(n_images=len(pairs['val']),empty_ground_truth_at_128=empty128,zero_fosp_predictions_native=zero_pred)
(out/'resolution_audit.json').write_text(json.dumps(summary,indent=2));print('RESOLUTION_AUDIT',json.dumps(summary),flush=True)


In [ ]:
# Adapt FoSp to SmokeSeg at native 512px; new baseline, not a novel method.
import gc,time,random,json,shutil
from pathlib import Path
import numpy as np,torch
import torch.nn.functional as F
from PIL import Image,ImageEnhance
ft=Path('/kaggle/working/day1_fosp_finetune');ft.mkdir(exist_ok=True)
cache.clear();gc.collect();torch.cuda.empty_cache()
model.requires_grad_(True);model.decode_head.G.requires_grad_(False)
ck=torch.load(weights,map_location='cpu',weights_only=True);model.load_state_dict(ck['state_dict'],strict=True);del ck
params=[p for p in model.parameters() if p.requires_grad]
groups=[{'params':[p],'lr':6e-5 if 'head' in n else 6e-6,'initial_lr':6e-5 if 'head' in n else 6e-6,'weight_decay':0 if ('norm' in n or p.ndim==1) else 0.01} for n,p in model.named_parameters() if p.requires_grad]
optim=torch.optim.AdamW(groups)
config_ft={'method':'FoSp warm-start14k finetune SmokeSeg','size':512,'batch':2,'accumulate':2,'threshold':0.4,'max_updates':2000,'budget_seconds':7200,'eval_every':250,'early_stopping_patience':4,'min_delta':0.001,'test_used':False,'split':'same date-group split as pilot','warning':'new-data baseline; not evidence for temporal novelty'}
(ft/'config.json').write_text(json.dumps(config_ft,indent=2));shutil.copy2('/kaggle/working/temporal_manifest.json',ft/'manifest.json');shutil.copy2('/kaggle/working/temporal_audit.json',ft/'data_audit.json')
random.seed(42);np.random.seed(42);torch.manual_seed(42)
start_ft=time.monotonic();step=0;best_ft=0.35583547014476075;hist_ft=[];stale_ft=0
meta={'img_shape':(512,512,3),'ori_shape':(512,512,3),'pad_shape':(512,512,3),'scale_factor':1.,'flip':False,'img_norm_cfg':{'mean':mean,'std':std,'to_rgb':True}}
def pair512(row,augment=False):
 with Image.open(row['image']) as im:im=im.convert('RGB').resize((512,512),Image.Resampling.BILINEAR)
 with Image.open(row['mask']) as m:target=Image.fromarray((np.asarray(m)>0).astype('uint8')).resize((512,512),Image.Resampling.NEAREST)
 if augment and random.random()<0.5:im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT);target=target.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
 if augment and random.random()<0.5:im=ImageEnhance.Brightness(im).enhance(random.uniform(.85,1.15))
 rgb=np.asarray(im,np.float32);x=torch.from_numpy(((rgb-mean)/std).transpose(2,0,1).copy());y=torch.from_numpy(np.array(target,dtype=np.int64))[None]
 return x,y
@torch.no_grad()
def val_ft():
 model.eval();state=np.random.get_state();rows=[];np.random.seed(123)
 try:
  for i,row in enumerate(pairs['val']):
   x,y=pair512(row);np.random.seed(123);z=model.encode_decode(x[None].cuda(),[{}]);p=F.interpolate(z,size=(512,512),mode='bilinear',align_corners=False)[0,0].sigmoid().cpu().numpy()>.4;t=y[0].numpy()>0;tp=int((p&t).sum());fp=int((p&~t).sum());fn=int((~p&t).sum());rows.append([2*tp/max(2*tp+fp+fn,1),tp/max(tp+fp+fn,1)])
   if (i+1)%100==0:print('FT_VAL_PROGRESS',i+1,flush=True)
 finally:np.random.set_state(state)
 return {'n_images':len(rows),'f1':float(np.mean(rows,0)[0]),'smoke_iou':float(np.mean(rows,0)[1]),'threshold':.4,'resolution':512}
def cpu_state():return {k:v.detach().cpu() for k,v in model.state_dict().items()}
def save_ft(best_scores=None):
 st=np.random.get_state();payload={'state_dict':cpu_state(),'optimizer':optim.state_dict(),'update':step,'best_f1':best_ft,'history':hist_ft,'config':config_ft,'stale':stale_ft,'rng':{'python':random.getstate(),'numpy':(st[0],st[1].tolist(),*st[2:]),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()}}
 save(payload,ft/'last.pth')
 if best_scores is not None:save({'state_dict':payload['state_dict'],'update':step,'validation':best_scores,'config':config_ft},ft/'best.pth')
 (ft/'history.json').write_text(json.dumps(hist_ft,indent=2));(ft/'status.json').write_text(json.dumps({'update':step,'best_f1':best_ft,'elapsed':time.monotonic()-start_ft}))
if (ft/'last.pth').is_file():
 c=torch.load(ft/'last.pth',map_location='cpu',weights_only=True);model.load_state_dict(c['state_dict']);optim.load_state_dict(c['optimizer']);step=c['update'];best_ft=c['best_f1'];hist_ft=c['history'];stale_ft=c['stale'];random.setstate(c['rng']['python']);ns=c['rng']['numpy'];np.random.set_state((ns[0],np.array(ns[1],dtype='uint32'),*ns[2:]));torch.set_rng_state(c['rng']['torch']);torch.cuda.set_rng_state_all(c['rng']['cuda']);del c
else:save_ft({'n_images':272,'f1':best_ft,'threshold':.4,'resolution':512})
print('FT_START',step,json.dumps(config_ft),flush=True)
try:
 while step<2000 and time.monotonic()-start_ft<6600:
  model.train();model.decode_head.G.eval();optim.zero_grad(set_to_none=True);loss_sum=0.
  for g in optim.param_groups:g['lr']=g['initial_lr']*min(1.,(step+1)/50)
  for _ in range(2):
   batch=[pair512(r,True) for r in random.sample(pairs['train'],2)];x=torch.stack([a[0] for a in batch]).cuda();y=torch.stack([a[1] for a in batch]).cuda();losses=model(img=x,img_metas=[meta]*2,gt_semantic_seg=y,return_loss=True);loss,_=model._parse_losses(losses)
   if not torch.isfinite(loss):raise FloatingPointError('Non-finite loss')
   (loss/2).backward();loss_sum+=float(loss.detach())/2
  torch.nn.utils.clip_grad_norm_(params,1.,error_if_nonfinite=True);optim.step();step+=1
  if step==1 or step%25==0:print('FT_UPDATE',step,'LOSS',round(loss_sum,5),'SECONDS',round(time.monotonic()-start_ft,1),'PEAK_GB',round(torch.cuda.max_memory_allocated()/2**30,2),flush=True)
  if step%250==0:
   scores=val_ft();hist_ft.append({'update':step,**scores});improve=scores['f1']>best_ft;significant=scores['f1']>best_ft+.001;best_ft=max(best_ft,scores['f1']);stale_ft=0 if significant else stale_ft+1;save_ft(scores if improve else None);print('FT_VALIDATION',json.dumps(hist_ft[-1]),flush=True)
   if stale_ft>=4:print('FT_EARLY_STOP',flush=True);break
  elif step%100==0:save_ft()
finally:
 save_ft();shutil.make_archive('/kaggle/working/forestwatch_day1_finetune','zip',ft);print('FT_SAVED',step,flush=True)
